# Task 1 Demo: predict music tags from a caption

Loads the fine-tuned BERT/DistilBERT multi-label tagger and runs end-to-end inference on a free-text music description. Run `src/data_musiccaps.py`, `src/train.py`, and `src/evaluate.py` first so the checkpoint and tag vocab exist.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import torch
from transformers import AutoTokenizer

from src.model import BertMultiLabelClassifier
from src.utils import ROOT, get_device, load_config

cfg = load_config()
device = get_device()

with open(ROOT / cfg["data"]["processed_dir"] / "tag_vocab.json") as f:
    tag_vocab = [e["tag"] for e in json.load(f)]

tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["name"])
model = BertMultiLabelClassifier(cfg["model"]["name"], len(tag_vocab)).to(device)
model.load_state_dict(torch.load(ROOT / cfg["paths"]["checkpoint_dir"] / "best_model.pt", map_location=device))
model.eval()
print(f"Loaded model on {device} with {len(tag_vocab)} tags")

In [ ]:
def predict_tags(caption: str, top_n: int = 8):
    enc = tokenizer(caption, max_length=cfg["data"]["max_length"], padding="max_length", truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
    probs = torch.sigmoid(logits)[0].cpu().numpy()
    top_idx = probs.argsort()[::-1][:top_n]
    return [(tag_vocab[i], float(probs[i])) for i in top_idx]


caption = "A slow, emotional piano ballad with soft female vocals and a melancholic string arrangement."
for tag, prob in predict_tags(caption):
    print(f"{tag:25s} {prob:.3f}")

In [ ]:
# Try your own caption
caption = "An upbeat electronic dance track with a driving bassline and punchy drums."
for tag, prob in predict_tags(caption):
    print(f"{tag:25s} {prob:.3f}")